## Schema enforcement & evolution

Delta enforces the table's schema on **every write**. By default, a write that introduces a new column **fails** — that's **schema enforcement**, and it's what stops a misbehaving upstream from silently growing the Cards table a `merchant_country` column.

When the new column *is* intentional, you opt in to **schema evolution**:

- **`mergeSchema = true`** on a write — adds the incoming DataFrame's missing columns to the table; existing rows get `NULL` there.
- **`overwriteSchema = true`** on an `INSERT OVERWRITE` — replaces the schema entirely. Destructive; use rarely.
- **`ALTER TABLE ... ADD COLUMNS (...)`** — explicit DDL, the most auditable option for production.

```python
(rogue_df.write
   .mode("append")
   .option("mergeSchema", "true")
   .saveAsTable("silver.card_transactions"))
```

Auto Loader (module 03) has its own evolution modes — `addNewColumns`, `rescue`, `failOnNewColumns`, `none` — but they're all built on this same Delta primitive.

**The rule that ships to the exam:** enforcement is **on by default**; evolution is **opt-in per write** or via explicit `ALTER TABLE`. Silent column additions are not a thing on Delta.